Init the graph store with documents and explore the graph store.

Before using this notebook be sure that the neo4j database `docs` exists (refer to the `init_graph_store_1.ipynb` notebook).


In [ ]:
import logging
import nest_asyncio
import sys
from llama_index.core import PropertyGraphIndex
from llama_index.core import Settings
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.embeddings.litellm import LiteLLMEmbedding
from llama_index.graph_stores.neo4j import Neo4jGraphStore
from llama_index.llms.litellm import LiteLLM
from pathlib import Path

from src.food_rag.utils import hummus_docify, llm_helper
from src.utils import load_util
from src.utils.jupyter_magic import as_markdown

nest_asyncio.apply()  # Required to run the notebook in Jupyter for disabling async operations in the notebook

logging.basicConfig(
    stream=sys.stdout, level=logging.INFO
)  # logging.INFO for less verbose output, logging.DEBUG for more verbose output

In [ ]:
# Output Paths
data_path = 'data'
embedding_path = data_path + '/embeddings/snowflake_l'

# Load key
api_key = load_util.load_file('config/api_key')

In [ ]:
# Init LLM
llm = LiteLLM(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model="openai/llama3.1"
)

embed_model = LiteLLMEmbedding(  # Doc see @ --> https://docs.litellm.ai/docs/providers/openai_compatible
    api_base="https://....",
    api_key=api_key,
    model_name="openai/Snowflake/snowflake-arctic-embed-l",
)

# Set Global Settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512

In [ ]:
# Load the current data
hummus_data = load_util.get_current_hummus(data_path)

## 1. Init Graph Store (or load if already exists)

In [ ]:
# Set up Neo4J params
username = "neo4j"
password = "neo4jfoodrag"
url = "bolt://localhost:7687"
database = "docs"

graph_store = Neo4jGraphStore(
    username=username,
    password=password,
    url=url,
    database=database,
)
storage_context = StorageContext.from_defaults(graph_store=graph_store)

In [ ]:
# Clear the graph store
graph_store.query("MATCH (n) DETACH DELETE n")

In [ ]:
# Convert the recipes to documents TODO for testing only use a small subset of the documents
recipe_documents = hummus_docify.recipes_to_docs(hummus_data['recipes_df'][:100])
#user_documents = hummus_docify.users_to_docs(hummus_data['users_df'][:20])
#review_documents = hummus_docify.reviews_to_docs(hummus_data['reviews_df'][:20])
#ingredient_documents = hummus_docify.ingredients_to_docs(hummus_data['ingredients_df'])
#food_product_documents = hummus_docify.food_products_to_docs(hummus_data['usda_df'])
food_on_documents = hummus_docify.food_on_to_docs(hummus_data['food_on_ontology'])[:20]

documents = recipe_documents + food_on_documents  # + user_documents + review_documents + ingredient_documents + food_product_documents

In [ ]:
# NOTE: can take a while!
# TODO snowflake_l context length is too small fro some documents, thus will fail with this model!
kg_index = PropertyGraphIndex.from_documents(
    documents,
    show_progress=True,
    storage_context=storage_context,
    max_triplets_per_chunk=2,
    timeout=60*60*30,
)

## 2. Store & load the index from existing Neo4J database

In [ ]:
# Save
path = Path(data_path, 'neo4j_kg', database)
load_util.ensure_dir(path)
kg_index.storage_context.persist(persist_dir=path)

In [ ]:
# Load
path = Path(data_path, 'neo4j_kg', database)
kg_index = load_index_from_storage(StorageContext.from_defaults(persist_dir=path))

## 3. Explore the Graph Store

In [ ]:
# Via natural language query using the graph store
retriever = kg_index.as_retriever()
response = retriever.retrieve("Can you recommend me a healthy recipe with chicken?")
as_markdown(response)

In [ ]:
# Via natural language query using the graph store
query_engine = kg_index.as_query_engine()
response = query_engine.query("Can you recommend me a healthy recipe with chicken?")
as_markdown(response)
print(llm_helper.get_serialized_entities(response))

In [ ]:
# Via natural language query using the graph store
query_engine = kg_index.as_chat_engine()
response = query_engine.query("Can you recommend me a healthy recipe with chicken?")
as_markdown(response)

### More exploration

In [ ]:
query_engine = kg_index.as_query_engine()

In [ ]:
response = query_engine.query("Why is the recipe 'Oven Baked White Rice' healthy and which NutriScore does it have?")
as_markdown(response)
print(llm_helper.get_serialized_entities(response))

In [ ]:
response = query_engine.query("Simplify the recipe 'Oven Baked White Rice', and also, how can I add some spice?")
as_markdown(response)
print(llm_helper.get_serialized_entities(response))